In [1]:
"""
Crawl repo GitHub được tạo từ 2023-10-01 đến 2026-09-30, có ít nhất 10 sao.

- REST Search API: lấy repo theo từng tháng, sắp xếp theo số sao giảm dần.
  Mỗi tháng lấy tối đa QUOTA dòng, để 12 000 dòng trải đều 36 tháng.
- GraphQL: lấy languages và kiểm tra README (search API không trả 2 trường này).
  Gom 50 repo mỗi request, chạy song song.
- Ghi theo lô 2000 dòng vào batches/, có checkpoint nên chạy lại sẽ tiếp tục.
- Cuối cùng gộp thành github_repos_complete.csv (đường dẫn notebook đang đọc).

Cách chạy:
    pip install requests
    export GITHUB_TOKEN=ghp_xxx        # Windows PowerShell: $env:GITHUB_TOKEN="ghp_xxx"
    python github_crawler.py
"""
import csv
import json
import math
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests

In [2]:
# ---------------- Cấu hình ----------------
TOKEN = os.environ.get("GITHUB_TOKEN")
if not TOKEN:
    raise SystemExit("Chưa đặt biến môi trường GITHUB_TOKEN")

OUT_DIR = Path("notebooks/data/raw")       # nơi notebook đọc input
BATCH_DIR = OUT_DIR / "batches"
CKPT = BATCH_DIR / "checkpoint.json"
FINAL = OUT_DIR / "github_repos_complete.csv"

START, END = date(2023, 10, 1), date(2026, 9, 30)
TARGET = 12_000
BATCH = 2_000
GQL_CHUNK = 50          # số repo mỗi request GraphQL
GQL_WORKERS = 4         # số request GraphQL chạy song song
SEARCH_INTERVAL = 2.1   # Search API: 30 request/phút cho tài khoản đã xác thực

COLUMNS = [
    "repo_id", "owner", "repo_name", "full_name", "primary_language", "languages",
    "description", "read_me", "topics", "license_key", "created_at", "updated_at",
    "pushed_at", "collected_at", "star_count", "forks_count", "open_issues_count",
    "size_kb", "has_issues", "has_wiki", "default_branch",
]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
session = requests.Session()
session.headers.update(HEADERS)
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

In [3]:
# ---------------- Giới hạn tốc độ ----------------
class Limiter:
    """Đảm bảo các request cách nhau ít nhất `interval` giây (an toàn khi nhiều thread)."""

    def __init__(self, interval):
        self.interval = interval
        self.lock = threading.Lock()
        self.last = 0.0

    def wait(self):
        with self.lock:
            delta = self.interval - (time.time() - self.last)
            if delta > 0:
                time.sleep(delta)
            self.last = time.time()


search_limiter = Limiter(SEARCH_INTERVAL)


def sleep_from_headers(r):
    """Tính thời gian chờ khi bị rate limit."""
    if r.headers.get("Retry-After"):
        return int(r.headers["Retry-After"]) + 1
    if r.headers.get("X-RateLimit-Reset"):
        return max(int(r.headers["X-RateLimit-Reset"]) - int(time.time()), 0) + 1
    return 60


def rest_get(url, params=None, limited=False):
    while True:
        if limited:
            search_limiter.wait()
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 429 or (r.status_code == 403 and "rate limit" in r.text.lower()):
            wait = sleep_from_headers(r)
            print(f"  Bị giới hạn tốc độ, chờ {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()


# ---------------- Bước 1: Search theo tháng ----------------
def month_windows(start, end):
    cur = start
    while cur <= end:
        nxt = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        yield cur, min(nxt - timedelta(days=1), end)
        cur = nxt


def iter_search(d1, d2):
    """Trả về các repo trong khoảng ngày, sắp theo sao giảm dần (tối đa 1000 kết quả/truy vấn)."""
    q = f"created:{d1.isoformat()}..{d2.isoformat()} stars:>=10 fork:false archived:false"
    for page in range(1, 11):
        data = rest_get(
            "https://api.github.com/search/repositories",
            {"q": q, "sort": "stars", "order": "desc", "per_page": 100, "page": page},
            limited=True,
        )
        items = data.get("items", [])
        if not items:
            return
        yield from items
        if len(items) < 100:
            return

In [4]:

# ---------------- Bước 2: GraphQL bổ sung languages và README ----------------
GQL = """
query($ids: [ID!]!) {
  nodes(ids: $ids) {
    ... on Repository {
      id
      languages(first: 30, orderBy: {field: SIZE, direction: DESC}) { nodes { name } }
      r1: object(expression: "HEAD:README.md") { __typename }
      r2: object(expression: "HEAD:README.rst") { __typename }
      r3: object(expression: "HEAD:README") { __typename }
    }
  }
}
"""


def enrich(node_ids):
    for attempt in range(5):
        r = session.post(
            "https://api.github.com/graphql",
            json={"query": GQL, "variables": {"ids": node_ids}},
            timeout=60,
        )
        data = r.json()
        if data.get("data") and not data.get("errors"):
            break
        if attempt == 4:
            raise RuntimeError(f"GraphQL lỗi: {data.get('errors') or r.text[:300]}")
        time.sleep(30)  # thường là do rate limit, chờ rồi thử lại

    out = {}
    for n in data["data"]["nodes"]:
        if not n:
            continue
        out[n["id"]] = {
            "languages": [x["name"] for x in n["languages"]["nodes"]],
            "read_me": any(n[k] for k in ("r1", "r2", "r3")),
        }
    return out


# ---------------- Ghi dữ liệu ----------------
def to_row(it, extra):
    lic = it.get("license") or {}
    return {
        "repo_id": it["id"],
        "owner": it["owner"]["login"],
        "repo_name": it["name"],
        "full_name": it["full_name"],
        "primary_language": it.get("language") or "",
        "languages": json.dumps(extra.get("languages", []), ensure_ascii=False),
        "description": it.get("description") or "",
        "read_me": extra.get("read_me", False),
        "topics": json.dumps(it.get("topics", []), ensure_ascii=False),
        "license_key": lic.get("key") or "",
        "created_at": it["created_at"],
        "updated_at": it["updated_at"],
        "pushed_at": it["pushed_at"],
        "collected_at": NOW,
        "star_count": it["stargazers_count"],
        "forks_count": it["forks_count"],
        "open_issues_count": it["open_issues_count"],
        "size_kb": it["size"],
        "has_issues": it["has_issues"],
        "has_wiki": it["has_wiki"],
        "default_branch": it.get("default_branch") or "",
    }


def write_csv(path, rows):
    # csv.DictWriter tự đặt dấu ngoặc kép cho ô có dấu phẩy, nên không còn lỗi ParserError
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS)
        w.writeheader()
        w.writerows(rows)

In [6]:
# ---------------- Pipeline chính ----------------
def main():
    BATCH_DIR.mkdir(parents=True, exist_ok=True)
    ckpt = json.loads(CKPT.read_text()) if CKPT.exists() else {"window": 0, "seen": [], "batches": 0}
    seen = set(ckpt["seen"])
    windows = list(month_windows(START, END))
    quota = math.ceil(TARGET / len(windows))  # số repo mỗi tháng
    print(f"{len(windows)} tháng, mỗi tháng {quota} repo, mục tiêu {TARGET}. Đã có {len(seen)} dòng.")

    buf = []
    pool = ThreadPoolExecutor(GQL_WORKERS)

    def flush(wi):
        if not buf:
            return
        node_ids = [it["node_id"] for it in buf]
        chunks = [node_ids[i:i + GQL_CHUNK] for i in range(0, len(node_ids), GQL_CHUNK)]
        extra = {}
        for res in pool.map(enrich, chunks):
            extra.update(res)

        rows = [to_row(it, extra.get(it["node_id"], {})) for it in buf]
        ckpt["batches"] += 1
        path = BATCH_DIR / f"batch_{ckpt['batches']:03d}.csv"
        write_csv(path, rows)

        ckpt["seen"] += [it["id"] for it in buf]
        ckpt["window"] = wi
        CKPT.write_text(json.dumps(ckpt))
        seen.update(it["id"] for it in buf)
        print(f"Đã ghi {path.name}: {len(rows)} dòng, tổng {len(seen)}/{TARGET}")
        buf.clear()

    def run_pass(start_window, per_window):
        for wi in range(start_window, len(windows)):
            if len(seen) + len(buf) >= TARGET:
                break
            d1, d2 = windows[wi]
            taken = 0
            for it in iter_search(d1, d2):
                if taken >= per_window or len(seen) + len(buf) >= TARGET:
                    break
                if it["id"] in seen or any(b["id"] == it["id"] for b in buf):
                    continue
                buf.append(it)
                taken += 1
                if len(buf) == BATCH:
                    flush(wi)
            print(f"  Xong tháng {d1:%Y-%m}: lấy {taken} repo")
        flush(len(windows) - 1)

    # Lượt 1: mỗi tháng lấy đủ phần quota
    run_pass(ckpt["window"], quota)

    # Lượt 2 (bù): nếu tháng nào thiếu repo đạt điều kiện, lấy thêm từ các tháng còn lại
    if len(seen) < TARGET:
        remaining = TARGET - len(seen)
        print(f"Thiếu {remaining} dòng, chạy lượt bù...")
        run_pass(0, math.ceil(remaining / len(windows)))

    pool.shutdown()

    # Gộp tất cả lô thành file cuối
    all_rows = []
    for p in sorted(BATCH_DIR.glob("batch_*.csv")):
        with open(p, encoding="utf-8-sig", newline="") as f:
            all_rows.extend(csv.DictReader(f))
    seen_ids, unique = set(), []
    for r in all_rows:
        if r["repo_id"] not in seen_ids:
            seen_ids.add(r["repo_id"])
            unique.append(r)
    unique = unique[:TARGET]
    write_csv(FINAL, unique)
    print(f"Hoàn tất: {len(unique)} dòng -> {FINAL}")


if __name__ == "__main__":
    main()

36 tháng, mỗi tháng 334 repo, mục tiêu 12000. Đã có 0 dòng.
  Xong tháng 2023-10: lấy 334 repo
  Xong tháng 2023-11: lấy 334 repo
  Xong tháng 2023-12: lấy 334 repo
  Xong tháng 2024-01: lấy 334 repo
  Xong tháng 2024-02: lấy 334 repo
Đã ghi batch_001.csv: 2000 dòng, tổng 2000/12000
  Xong tháng 2024-03: lấy 334 repo
  Xong tháng 2024-04: lấy 334 repo
  Xong tháng 2024-05: lấy 334 repo
  Xong tháng 2024-06: lấy 334 repo
  Xong tháng 2024-07: lấy 334 repo
  Xong tháng 2024-08: lấy 334 repo
Đã ghi batch_002.csv: 2000 dòng, tổng 4000/12000
  Xong tháng 2024-09: lấy 334 repo
  Xong tháng 2024-10: lấy 334 repo
  Xong tháng 2024-11: lấy 334 repo
  Xong tháng 2024-12: lấy 334 repo
  Xong tháng 2025-01: lấy 334 repo
  Xong tháng 2025-02: lấy 334 repo
Đã ghi batch_003.csv: 2000 dòng, tổng 6000/12000
  Xong tháng 2025-03: lấy 334 repo
  Xong tháng 2025-04: lấy 334 repo
  Xong tháng 2025-05: lấy 334 repo
  Xong tháng 2025-06: lấy 334 repo
  Xong tháng 2025-07: lấy 334 repo
  Xong tháng 2025-08: l